PYDANTIC

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    director:str=Field(description="the director of the movie")
    year:int=Field(description="the year movie released")
    ratingds:float=Field(description="ratings of the movie")

In [4]:
model_with_structured = model.with_structured_output(Movie)

In [5]:
model_with_structured.invoke("tell me about movie Interstellar") #response follows the structured schema

Movie(title='Interstellar', director='Christopher Nolan', year=2014, ratingds=8.2)

In [6]:
model.invoke("tell me about movie Interstellar")

AIMessage(content='"Interstellar" is a 2014 science fiction film directed by Christopher Nolan, based on the 2007 short story "The Last Question" by Isaac Asimov and an original screenplay by Nolan. The film stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Michael Caine, and Mackenzie Foy.\n\n**Plot**\n\nThe film takes place in a dystopian future where Earth is facing an impending environmental disaster due to a severe food shortage. Climate change has made most of the planet\'s crops uninhabitable, and humanity is on the brink of extinction.\n\nA team of scientists and engineers, led by Professor Brand (Michael Caine), has been working on a top-secret project to find a new habitable planet for humanity. They have discovered three potential planets that could support human life, but they are too far away to reach with current technology.\n\nThe team comes up with a plan to send a spaceship, the Endurance, through a wormhole in search of a new home. The crew of the Endurance 

MESSAGE OUTPUT WITH PARSED STRUCTURE

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="the title of the movie")
    director:str=Field(...,description="the director of the movie")
    year:int=Field(...,description="the year movie released")
    ratingds:float=Field(...,description="ratings of the movie")
    # explanation: str = Field(description="Explain about the movie")
    
    
model_with_structured = model.with_structured_output(Movie,include_raw= True)

model_with_structured.invoke("tell me about movie Interstellar")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jpyrsm0g6', 'function': {'arguments': '{"director":"Christopher Nolan","ratingds":8.2,"title":"Interstellar","year":2014}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 285, 'total_tokens': 319, 'completion_time': 0.036143029, 'completion_tokens_details': None, 'prompt_time': 0.020756377, 'prompt_tokens_details': None, 'queue_time': 0.048156292, 'total_time': 0.056899406}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2d90-644c-7d02-91f0-2b4a025d28b7-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'ratingds': 8.2, 'title': 'Interstellar', 'year': 2014}, 'id': 'jpyrsm0g6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 285, 'output_token

In [10]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    cast:list[Actor]
    genre:list[str]
    director:str
    year:int
    budget:float | None = Field(None,description="Budget of the movie")
    
model_with_structured = model.with_structured_output(MovieDetails)

model_with_structured.invoke("tell me about movie Interstellar")

MovieDetails(title='Interstellar', cast=[Actor(name='Matthew McConaughey', role='Cooper'), Actor(name='Anne Hathaway', role='Brand'), Actor(name='Jesse Plemons', role='Case'), Actor(name='Michael Caine', role='Professor Brand'), Actor(name='Casey Affleck', role='Tom Cooper'), Actor(name='Dave Power', role='Hannibal'), Actor(name='Isaac England', role='Murph'), Actor(name='Mackenzie Foy', role='Young Murph'), Actor(name='Jessica Chastain', role='Young Brand'), Actor(name='Matt Damon', role=' Mann')], genre=['Science Fiction', 'Adventure', 'Mystery'], director='Christopher Nolan', year=2014, budget=None)

TYPEDICT - NO NEED RUNTIME VALIDATION

In [11]:
from typing_extensions import TypedDict,Annotated 


class Movie(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"the title of the movie"]
    director: Annotated[str,...,"the director of the movie"]
    year: Annotated[int,...,"the year movie released"]
    ratings: Annotated[float,...,"ratings of the movie"]
    # explanation: str = Annotated(description="Explain about the movie")
    
    
model_with_structured = model.with_structured_output(Movie)

model_with_structured.invoke("tell me about movie Interstellar")

{'director': 'Christopher Nolan',
 'ratings': 8.1,
 'title': 'Interstellar',
 'year': 2014}

In [21]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma:2b"
)

In [22]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    cast:list[Actor]
    genre:list[str]
    year:int
    budget: float | None = Field(None,description="Budget in millions USD")
    
model_with_structured = llm.with_structured_output(MovieDetails)

model_with_structured.invoke("tell me about movie Interstellar")

{'title': 'Interstellar',
 'cast': [{'name': 'Christopher Nolan', 'role': 'Cooper'},
  {'name': 'Matt Damon', 'role': 'Leonard Hofstadter'},
  {'name': 'Anne Hathaway', 'role': 'Brand'}],
 'genre': ['Science fiction', 'Drama'],
 'year': 2014,
 'budget': 160.81571428571436}

In [23]:
model.profile

{'max_input_tokens': 8192,
 'max_output_tokens': 8192,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

In [25]:
llm.profile

DATA CLASSES

In [4]:
##PYDANTIC , AGENTS 
from langchain_ollama import ChatOllama
from pydantic import BaseModel,Field
from langchain.agents import create_agent

# llm = ChatOllama(
#     model="gemma:2b"
# )

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent( 
                     model = model,
                     response_format= ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='fd5e10c2-056d-4ebd-a3f7-84609dc4b8fc'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q56n373j0', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 288, 'total_tokens': 319, 'completion_time': 0.167622803, 'completion_tokens_details': None, 'prompt_time': 0.094134478, 'prompt_tokens_details': None, 'queue_time': 0.05238773, 'total_time': 0.261757281}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2e1b-bdc7-7363-b720-a444601282f1-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@

In [5]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [7]:
##TypedDict , AGENTS 
from typing_extensions import TypedDict

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str #"The name of the person"
    email: str #"The email address of the person"
    phone: str #"The phone number of the person"

agent = create_agent( 
                     model = model,
                     response_format= ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [8]:
##Dataclass, AGENTS 
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str #"The name of the person"
    email: str #"The email address of the person"
    phone: str #"The phone number of the person"

agent = create_agent( 
                     model = model,
                     response_format= ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')